In [1]:
import json
import shutil
from pathlib import Path

PROJECT = Path.home() / "projects/knee-oa-classification"
HF_DIR = PROJECT / "hf_model"
HF_DIR.mkdir(exist_ok=True)

for name in ["exp1_augmentation", "exp3_aug_wider"]:
    shutil.copy(PROJECT / "models" / f"{name}.keras", HF_DIR / f"{name}.keras")
shutil.copy(PROJECT / "src/preprocessing.py", HF_DIR / "preprocessing.py")
shutil.copy(PROJECT / "MODEL_CARD.md", HF_DIR / "README.md")       # Hugging Face shows README.md as the card

config = {
    "version": "1.0",
    "members": ["exp1_augmentation.keras", "exp3_aug_wider.keras"],
    "input": {"shape": [224, 224], "channels": 1, "dtype": "uint8", "scale": "divide by 255"},
    "decision_rule": "argmax(mean_softmax / train_prior)",
    "train_counts_by_grade": [2286, 1046, 1516, 757, 173],
    "grade_names": ["Healthy", "Doubtful", "Minimal", "Moderate", "Severe"],
    "gradcam_layer": "block4_relu2",
}
(HF_DIR / "pipeline_config.json").write_text(json.dumps(config, indent=2))

for f in sorted(HF_DIR.iterdir()):
    print(f"{f.name:28s} {f.stat().st_size / 1e6:6.2f} MB")

README.md                      0.01 MB
exp1_augmentation.keras       14.23 MB
exp3_aug_wider.keras          31.81 MB
pipeline_config.json           0.00 MB
preprocessing.py               0.00 MB


In [2]:
from huggingface_hub import HfApi

api = HfApi()
user = api.whoami()["name"]
repo_id = f"{user}/knee-oa-kl-grading"

api.create_repo(repo_id, repo_type="model", private=True, exist_ok=True)
api.upload_folder(folder_path=HF_DIR, repo_id=repo_id, repo_type="model",
                  commit_message="v1.0: ensemble of two custom CNNs + preprocessing")
print(f"Uploaded: https://huggingface.co/{repo_id}")

/home/afzal/miniconda3/envs/knee-oa/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Uploaded: https://huggingface.co/afzal2003/knee-oa-kl-grading


In [3]:
import sys
import numpy as np
import keras
from huggingface_hub import hf_hub_download

sys.path.insert(0, str(PROJECT / "src"))
import data
import evaluation as ev

models_hub = [keras.models.load_model(hf_hub_download(repo_id, f)) for f in config["members"]]
X_val, y_val = data.load_split("val")
val_ds = data.make_dataset(X_val, y_val, batch_size=64)

probs = np.mean([m.predict(val_ds, verbose=0) for m in models_hub], axis=0)
prior = np.array(config["train_counts_by_grade"]) / sum(config["train_counts_by_grade"])
m = ev.compute_metrics(y_val, (probs / prior).argmax(axis=1))
print(f"Models downloaded from the Hub: validation balanced accuracy {m['balanced_accuracy']:.4f} (expect 0.6467)")

I0000 00:00:1791140974.348052  100640 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791140974.626384  100640 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791140976.250218  100640 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791140982.912250  100640 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5560 MB mem

Models downloaded from the Hub: validation balanced accuracy 0.6467 (expect 0.6467)
